In [1]:
import boto3

In [2]:
s3 = boto3.client("s3")

In [3]:
response = s3.list_objects_v2(Bucket="llm-finetune-dataset-ratul", Prefix="datasets /")

In [4]:
for obj in response.get("Contents", []):
    print(obj)

In [5]:
for obj in response.get("Contents", []):
    print(obj["Key"])

In [6]:
dataset_path = "s3://llm-finetune-dataset-ratul/datasets/pharma_instruction_data.csv"

In [7]:
from datasets import load_dataset

In [ ]:
! which python

In [ ]:
# !conda install -y -c conda-forge pyarrow datasets

In [8]:
dataset = load_dataset("csv", data_files={"train":dataset_path}, split="train")

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


In [9]:
dataset

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 6
})

In [10]:
print(dataset[0])

{'instruction': 'Summarize the purpose of the drug.', 'input': 'Drug: Metformin. Used for management of type 2 diabetes.', 'output': 'Metformin helps control blood sugar levels in people with type 2 diabetes.'}


In [11]:
def format_example(example):
    prompt = f"### Instruction: \n{example['instruction']} \n###Input: \n{example['input']}\n### Output: \n{example['output']}"
    return {"text": prompt}


In [12]:
dataset = dataset.map(format_example)

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

In [13]:
dataset

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 6
})

In [14]:
dataset['text'][0]

'### Instruction: \nSummarize the purpose of the drug. \n###Input: \nDrug: Metformin. Used for management of type 2 diabetes.\n### Output: \nMetformin helps control blood sugar levels in people with type 2 diabetes.'

In [ ]:
# !pip install torch

In [ ]:
#! pip install peft

In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

In [16]:
model_id = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [17]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [18]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [19]:
def tokenize_fn(example):
    tokens = tokenizer(example["text"], truncation = True, padding = "max_length", max_length = 512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [20]:
tokenized = dataset.map(tokenize_fn, batched = True)

Parameter 'function'=<function tokenize_fn at 0x7f25154c2a20> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

In [21]:
from peft import LoraConfig, get_peft_model, TaskType

In [22]:
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 